In [1]:
from platform import python_version
print(python_version())

3.11.14


### BayesPrism

https://github.com/Danko-Lab/BayesPrism


#### **Bayesian cell Proportion Reconstruction** Inferred using Statistical Marginalization (BayesPrism):

A Fully Bayesian Inference of Tumor Microenvironment composition and gene expression

BayesPrism consists of 
- the deconvolution modules and 
- the embedding learning module. 

The **deconvolution module** models a prior from cell type-specific expression profiles from scRNA-seq to jointly estimate the posterior distribution of cell type composition and cell type-specific gene expression from bulk RNA-seq expression of tumor (or non-tumor) samples. 

The **embedding learning** module uses Expectation-maximization (EM) to approximate the tumor expression using a linear combination of malignant gene programs while conditional on the inferred expression and fraction of non-malignant cells estimated by the deconvolution module.


#### Ref

Cell type and gene expression deconvolution with BayesPrism enables Bayesian integrative analysis across bulk and single-cell RNA sequencing in oncology

Chu, T. et al. & Danko, C.G.

https://www.nature.com/articles/s43018-022-00356-3


#### Concepts (paper)

Two layers of information are critical for understanding tumor composition: (1) the proportion of each cell type and (2) the levels of gene expression in each cell type. The rise of single-cell RNA sequencing (scRNA-seq) technologies has recently enabled direct, genome-wide measurement of the transcriptome in individual
cells within the TME and characterization of their heterogeneity. However, the cost of scRNA-seq and requirements for high-quality tissue limit the number of patient samples that can be assayed9. Moreover, **scRNA-seq is susceptible to technical biases in cell capture** 9 , which confound the recovery of cell type composition.

### Github

https://github.com/Danko-Lab/BayesPrism

- tutorial_deconvolution.html
- tutorial_embedding_learning.html



In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

verbose=True

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)


-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"


In [8]:
verbose=False
force=False

imax_tumor=200
imax_normal=100

df_tumor, df_normal, df_gtex_ctrl = cbio.calc_file_expression_tumor_normal_gtex(
            imax_tumor=imax_tumor, imax_normal=imax_normal, force=force, verbose=verbose)

print(df_tumor.shape[1], df_normal.shape[1], df_gtex_ctrl.shape[1])


52 23 0


In [9]:
df_tumor.head(3)

,geneid,symbol,biotype,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,...,T-C3L-03632,T-C3N-02589,T-C3N-03006,T-C3N-02996,T-C3N-03754,T-C3L-03639,T-C3L-02606,T-C3N-03665,T-C3N-03173,T-C3N-02696
0,ENSG00000000003,TSPAN6,protein_coding,1486,2083,1558,546,1208,648,896,...,730,1539,1671,1150,1709,2294,1693,368,1224,1882
1,ENSG00000000005,TNMD,protein_coding,12,97,15,1,14,2,5,...,50,7,11,12,14,31,5,4,8,9
2,ENSG00000000419,DPM1,protein_coding,1330,1521,1499,986,1388,974,649,...,949,1635,1768,913,1183,1592,1175,459,1364,1464


In [10]:
df_normal.head(3)

,geneid,symbol,biotype,N-C3L-04072,N-C3L-00589,N-C3L-03123,N-C3L-04080,N-C3L-00640,N-C3N-01719,N-C3L-07033,...,N-C3N-01899,N-C3N-00517,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696
0,ENSG00000000003,TSPAN6,protein_coding,1633,1302,1079,1367,896,1188,1275,...,791,1695,891,1063,1261,1821,554,1244,977,1576
1,ENSG00000000005,TNMD,protein_coding,0,8,9,3,4,7,0,...,1,2,0,7,2,6,1,1,3,39
2,ENSG00000000419,DPM1,protein_coding,1281,937,655,956,1000,1504,1087,...,619,1075,703,601,1059,1289,342,773,676,783


In [11]:
df_gtex_ctrl.head(3)

""


### All samples

In [12]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file


>> dfn_tumor (60616, 134)
>> dfn_normal (60616, 25)


In [13]:
dfn_tumor.head(3)

,geneid,symbol,biotype,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,...,T-TCGA-HZ-7289,T-TCGA-3A-A9IN,T-TCGA-3A-A9IS,T-TCGA-2L-AAQM,T-TCGA-3A-A9IR,T-TCGA-3A-A9IV,T-TCGA-3A-A9IO,T-TCGA-2J-AABT,T-TCGA-H6-A45N,T-TCGA-3A-A9IJ
0,ENSG00000000003,TSPAN6,protein_coding,1486,2083,1558,546,1208,648,896,...,2506,442,38,299,77,158,394,659,1294,395
1,ENSG00000000005,TNMD,protein_coding,12,97,15,1,14,2,5,...,2,172,4,2,1,2,14,3,3,0
2,ENSG00000000419,DPM1,protein_coding,1330,1521,1499,986,1388,974,649,...,1638,940,1372,1008,1493,1059,922,839,717,1034


In [14]:
dfn_normal.head(3)

,geneid,symbol,biotype,N-C3L-04072,N-C3L-00589,N-C3L-03123,N-C3L-04080,N-C3L-00640,N-C3N-01719,N-C3L-07033,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
0,ENSG00000000003,TSPAN6,protein_coding,1633,1302,1079,1367,896,1188,1275,...,891,1063,1261,1821,554,1244,977,1576,3738,369
1,ENSG00000000005,TNMD,protein_coding,0,8,9,3,4,7,0,...,0,7,2,6,1,1,3,39,4,5
2,ENSG00000000419,DPM1,protein_coding,1281,937,655,956,1000,1504,1087,...,703,601,1059,1289,342,773,676,783,1532,1023


In [15]:
# cbio.plot_boxplot_expression(dfn_tumor, do_log10=True, title = "Expression across tumor samples")

In [16]:
# cbio.plot_boxplot_expression(dfn_normal, do_log10=True, title = "Expression across normal samples")

### Prism - development

In [17]:
import anndata as ad

from libs.prism_lib import PRISM

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



(PosixPath('/home/flavio/uv/perturb_agent/data/single_cell'), True)

In [18]:
fname="count-matrix.txt"
prism.load_and_view_matrix_txt(fname=fname, nrows=10)

field counts: {"'\\t'": 0, "','": 0, "' '": 57530, "';'": 0} -> sep = ' '
first 10 rows x 5 cols:
Empty DataFrame
Columns: []
Index: [AL627309.1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0

True

In [19]:
adata = prism.load_matrix(fname=fname)
fname_celltype="all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype)

all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 2447
Acinar cell            1935
Endocrine cell          729
Name: count, dtype: int64


In [20]:
rep = prism.check_reference(adata=adata)
assert rep["X_looks_like_raw_counts"], rep["problems"]

In [21]:
ref, s2t = prism.pseudobulk_reference(adata)

In [22]:
df_bulk, meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata)
gene_subset=prism.select_genes(ref)


In [23]:
meta_desc = dict(reference="Peng2019_CRA001160",
              cohorts=["TCGA-PAAD", "CPTAC3"],
              strand="unstranded",
              method="InstaPrism")

force=False
verbose=True

res = prism.run_bayesprism(df_bulk=df_bulk, meta_desc=meta_desc, 
                           ref=ref, state_to_type=s2t, 
                           gene_subset=gene_subset,
                           force=force, verbose=verbose)

Loaded /home/flavio/uv/perturb_agent/data/single_cell/deconv.h5ad (6.8 MB)


### Cell state (Tutorial: bulk RNA-seq deconvolution using BayesPrism)

Please make sure that all cell states contain a reasonable number of cells, e.g. >20 or >50, so that their profile can be represented accurately.

What to supply for cell.state.labels and cell.type.labels? The definition of cell type and cell state can be somewhat arbitrary (similar to the issue of assigning cell types for scRNA-seq) and depends on the question of interest. Their definitions depend on the granularity we aim at and the confidence of the cell.type.labels in scRNA-seq data. Usually, a good rule of thumb is as follows. 1) Define cell types as the cluster of cells having a sufficient number of significantly differentially expressed genes than other cell types, e.g., greater than 50 or even 100. For clusters that are too similar in transcription, we recommend treating them as cell states, which will be summed up before the final Gibbs sampling. Therefore, cell states are often suitable for cells that form a continuum on the phenotypic manifold rather than distinct clusters. 2) Define multiple cell states for cell types of significant heterogeneity, such as malignant cells, and of interest to deconvolve their transcription.

In [50]:
s2t

cell_state
Fibroblast cell          Fibroblast cell
Stellate cell              Stellate cell
Macrophage cell          Macrophage cell
Endothelial cell        Endothelial cell
T cell                            T cell
B cell                            B cell
Ductal cell type 2             malignant
Endocrine cell            Endocrine cell
Ductal cell type 1    Ductal cell type 1
Acinar cell                  Acinar cell
Name: cell_type, dtype: object

In [42]:
dic = res.__dict__
dic.keys()

dict_keys(['theta', 'theta_stage1', 'theta_type', 'tumor_purity', 'genes', 'Z', 'states'])

In [43]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [44]:
res.tumor_purity

T-C3L-02890       0.102
T-C3L-03635       0.213
T-C3L-02701       0.193
T-C3L-04072       0.530
T-C3L-00589       0.430
                  ...  
N-C3L-02606       0.000
N-C3N-03173       0.930
N-C3N-02696       0.000
N-TCGA-H6-8124    0.087
N-TCGA-H6-A45N    0.000
Name: tumor_purity, Length: 153, dtype: float64

In [45]:
res.theta_type

,Acinar cell,B cell,Ductal cell type 1,Endocrine cell,Endothelial cell,Fibroblast cell,Macrophage cell,Stellate cell,T cell,malignant
T-C3L-02890,1.195e-01,6.756e-03,1.085e-01,6.223e-137,0.035,0.594,0.024,1.027e-02,1.286e-67,0.102
T-C3L-03635,0.000e+00,2.540e-03,8.157e-82,1.121e-79,0.035,0.726,0.017,7.818e-03,1.260e-167,0.213
T-C3L-02701,0.000e+00,8.332e-29,4.128e-156,1.838e-02,0.020,0.740,0.027,7.710e-04,3.798e-241,0.193
T-C3L-04072,3.138e-03,1.548e-02,2.339e-02,0.000e+00,0.038,0.292,0.081,1.838e-02,2.491e-140,0.530
T-C3L-00589,4.390e-02,2.909e-03,1.155e-02,9.594e-279,0.042,0.378,0.046,4.566e-02,9.531e-124,0.430
...,...,...,...,...,...,...,...,...,...,...
N-C3L-02606,0.000e+00,0.000e+00,0.000e+00,0.000e+00,0.000,0.000,0.000,0.000e+00,0.000e+00,0.000
N-C3N-03173,3.245e-105,1.669e-03,1.471e-02,0.000e+00,0.008,0.004,0.002,4.027e-02,9.874e-82,0.930
N-C3N-02696,0.000e+00,0.000e+00,0.000e+00,0.000e+00,0.000,0.000,0.000,0.000e+00,0.000e+00,0.000
N-TCGA-H6-8124,4.016e-02,2.273e-09,1.479e-01,4.980e-03,0.035,0.582,0.088,1.552e-02,6.081e-142,0.087


In [46]:
res.theta

,Fibroblast cell,Stellate cell,Macrophage cell,Endothelial cell,T cell,B cell,Ductal cell type 2,Endocrine cell,Ductal cell type 1,Acinar cell
T-C3L-02890,0.594,1.027e-02,0.024,0.035,1.286e-67,6.756e-03,0.102,6.223e-137,1.085e-01,1.195e-01
T-C3L-03635,0.726,7.818e-03,0.017,0.035,1.260e-167,2.540e-03,0.213,1.121e-79,8.157e-82,0.000e+00
T-C3L-02701,0.740,7.710e-04,0.027,0.020,3.798e-241,8.332e-29,0.193,1.838e-02,4.128e-156,0.000e+00
T-C3L-04072,0.292,1.838e-02,0.081,0.038,2.491e-140,1.548e-02,0.530,0.000e+00,2.339e-02,3.138e-03
T-C3L-00589,0.378,4.566e-02,0.046,0.042,9.531e-124,2.909e-03,0.430,9.594e-279,1.155e-02,4.390e-02
...,...,...,...,...,...,...,...,...,...,...
N-C3L-02606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N-C3N-03173,0.004,4.027e-02,0.002,0.008,9.874e-82,1.669e-03,0.930,0.000e+00,1.471e-02,3.245e-105
N-C3N-02696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N-TCGA-H6-8124,0.582,1.552e-02,0.088,0.035,6.081e-142,2.273e-09,0.087,4.980e-03,1.479e-01,4.016e-02


In [48]:
res.Z.shape

(153, 10, 1604)

In [49]:
len(res.genes)

1604

### Prism

load_cra001160.py  

Convert the Peng 2019 GSA deposit into an AnnData ready for
`paad_deconv.pseudobulk_reference()`.

Input (from ftp://download.big.ac.cn/gsa/CRA001160/):
- count-matrix.txt   2.77 GB dense TSV, genes x cells
- all_celltype.txt   2.1 MB, per-cell annotation

The matrix is dense text: ~20k genes x ~57k cells is ~1.1e9 values, which is 10-13 GB as a dense float array but well under 1 GB as CSR, since scRNA counts are >90% zeros. So it is parsed in row chunks and sparsified incrementally -- never materialised dense.

```Bash
lftp -e "cls -l; quit" ftp://download.big.ac.cn/gsa/CRA001160/

lftp -e "pget -n 8 -c count-matrix.txt; \
         get all_celltype.txt; get md5sum.txt; quit"      ftp://download.big.ac.cn/gsa/CRA001160/
```


In [24]:
!python -m ipykernel install --user --name renv --display-name "Python (renv)"

Installed kernelspec renv in /home/flavio/.local/share/jupyter/kernels/renv


### Single-cell quality: MAESTRO

MAESTRO (Model-based AnalysEs of Single-cell Transcriptome and RegulOme) is a Snakemake-based pipeline that processes single-cell RNA-seq and ATAC-seq data from raw FASTQ files through alignment, quality control, cell filtering, clustering, and cell-type annotation.

https://liulab-dfci.github.io/MAESTRO/

### TISCH2

https://tisch.compbio.cn/gallery/?cancer=PAAD&celltype=Acinar&celltype=Ductal&species=Human&treatment=None&primary=Primary


ref: Peng J, Sun BF, Chen CY, Zhou JY, Chen YS, Chen H, Liu L, Huang D, Jiang J, Cui GS, Yang Y, Wang W, Guo D, Dai M, Guo J, Zhang T, Liao Q, Liu Y, Zhao YL, Han DL, Zhao Y, Yang YG, Wu W. Single-cell RNA-seq highlights intra-tumoral heterogeneity and malignant progression in pancreatic ductal adenocarcinoma. Cell Res. 2019 Sep;29(9):725-738. doi: 10.1038/s41422-019-0195-y. Epub 2019 Jul 4. Erratum in: Cell Res. 2019 Sep;29(9):777. doi: 10.1038/s41422-019-0212-1. PMID: 31273297; PMCID: PMC6796938.

In [25]:
import anndata as ad

# sc_ref = ad.read_h5ad("PAAD_Peng2019_annotated.h5ad")   # raw counts in .X

###  nnls_deconvolve()

It's the baseline cross-check — deliberately not part of the main path. It's there so you can ask "is my reference sane?" without trusting the engine you're validating.

What it computes. For each sample independently, it solves

min_w  ||Φᵀw − b_s||²    subject to  w ≥ 0

where b_s is the sample's CPM vector and Φᵀ is the genes × states signature matrix (each state row CPM-normalized). Then it rescales w to sum to 1. That's the dtangle / CIBERSORT family: linear unmixing under a Gaussian loss.

How it differs from prism_em, which matters more than it looks:

|	      | nnls_deconvolve	 |prism_em |
|---------|------------------|---------|
| loss	  | L2 on CPM	| multinomial on counts |
| gene weighting | high-expression genes dominate | Poisson variance weights each gene naturally |
| simplex	|  imposed post hoc by rescaling | enforced every iteration |
| malignant reference | fixed | sample-specific (stage 2) |

The L2-on-CPM part is the substantive difference. A gene at 5,000 CPM contributes ~10⁶× more residual than one at 5 CPM, so NNLS is effectively fit on a few dozen highly-expressed genes regardless of how informative they are. The multinomial likelihood weights each gene by its own expected count, which is the correct variance model for counts.

Why you'd actually run it. Concordance is a reference-quality diagnostic, and the pattern of disagreement is informative:

Stromal/immune states agree closely (Spearman > 0.8) → reference is fine
Stromal/immune states disagree → your phi is broken, or select_genes picked protocol-driven genes; fix that before interpreting anything
Malignant compartment disagrees while the rest agrees → expected and good. That's stage 2 doing its job. If NNLS and EM agree on purity, update_malignant_reference isn't contributing and PDAC classical/basal heterogeneity is still leaking into the stromal fractions

To wire it in:

In [26]:
th_nnls = prism.nnls_deconvolve(df_bulk, ref, genes=res.genes)
th_nnls

cell_state,Fibroblast cell,Stellate cell,Macrophage cell,Endothelial cell,T cell,B cell,Ductal cell type 2,Endocrine cell,Ductal cell type 1,Acinar cell
T-C3L-02890,0.808,0.061,0.000,0.00,0.0,0.000,0.000,0.0,0.000,0.131
T-C3L-03635,0.962,0.038,0.000,0.00,0.0,0.000,0.000,0.0,0.000,0.000
T-C3L-02701,0.911,0.089,0.000,0.00,0.0,0.000,0.000,0.0,0.000,0.000
T-C3L-04072,0.749,0.145,0.000,0.00,0.0,0.000,0.082,0.0,0.000,0.024
T-C3L-00589,0.680,0.126,0.005,0.00,0.0,0.000,0.113,0.0,0.000,0.076
...,...,...,...,...,...,...,...,...,...,...
N-C3L-02606,0.245,0.019,0.000,0.00,0.0,0.000,0.000,0.0,0.000,0.736
N-C3N-03173,0.096,0.234,0.051,0.00,0.0,0.025,0.224,0.0,0.330,0.039
N-C3N-02696,0.375,0.031,0.000,0.00,0.0,0.000,0.000,0.0,0.020,0.574
N-TCGA-H6-8124,0.808,0.107,0.000,0.00,0.0,0.000,0.000,0.0,0.023,0.061


In [27]:
th_em   = res.theta
th_em

,Fibroblast cell,Stellate cell,Macrophage cell,Endothelial cell,T cell,B cell,Ductal cell type 2,Endocrine cell,Ductal cell type 1,Acinar cell
T-C3L-02890,0.594,1.027e-02,0.024,0.035,1.286e-67,6.756e-03,0.102,6.223e-137,1.085e-01,1.195e-01
T-C3L-03635,0.726,7.818e-03,0.017,0.035,1.260e-167,2.540e-03,0.213,1.121e-79,8.157e-82,0.000e+00
T-C3L-02701,0.740,7.710e-04,0.027,0.020,3.798e-241,8.332e-29,0.193,1.838e-02,4.128e-156,0.000e+00
T-C3L-04072,0.292,1.838e-02,0.081,0.038,2.491e-140,1.548e-02,0.530,0.000e+00,2.339e-02,3.138e-03
T-C3L-00589,0.378,4.566e-02,0.046,0.042,9.531e-124,2.909e-03,0.430,9.594e-279,1.155e-02,4.390e-02
...,...,...,...,...,...,...,...,...,...,...
N-C3L-02606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N-C3N-03173,0.004,4.027e-02,0.002,0.008,9.874e-82,1.669e-03,0.930,0.000e+00,1.471e-02,3.245e-105
N-C3N-02696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N-TCGA-H6-8124,0.582,1.552e-02,0.088,0.035,6.081e-142,2.273e-09,0.087,4.980e-03,1.479e-01,4.016e-02


In [28]:
conc = pd.DataFrame({
    "spearman": {k: th_em[k].corr(th_nnls[k], method="spearman") for k in res.states},
    "mean_em":   th_em.mean(),
    "mean_nnls": th_nnls.mean(),
})
conc["bias"] = conc.mean_em - conc.mean_nnls
print(conc.sort_values("spearman"))

                    spearman    mean_em  mean_nnls       bias
T cell                 0.129  5.531e-04  1.139e-04  4.393e-04
Macrophage cell        0.148  2.493e-02  2.083e-02  4.108e-03
Endothelial cell       0.174  2.517e-02  3.891e-03  2.128e-02
B cell                 0.214  4.501e-03  3.783e-03  7.176e-04
Endocrine cell         0.315  3.192e-02  1.701e-03  3.022e-02
Stellate cell          0.503  2.686e-02  7.063e-02 -4.377e-02
Ductal cell type 1     0.590  1.988e-02  3.296e-02 -1.307e-02
Ductal cell type 2     0.854  3.416e-01  8.464e-02  2.569e-01
Fibroblast cell        0.929  3.583e-01  5.633e-01 -2.050e-01
Acinar cell            0.961  1.663e-01  2.182e-01 -5.188e-02


Pass genes=res.genes explicitly. The default (all shared genes) lets housekeeping genes drive the L2 fit and the comparison stops being meaningful.

One caveat before you trust a low correlation. 

NNLS handles collinear reference profiles badly 
— with Ductal cell type 1 vs type 2, or iCAF vs myCAF, 

it tends to zero one of the pair out arbitrarily per sample, so its per-state estimates are unstable even when the aggregate is right. 

Check the conditioning first:

In [30]:
g = df_bulk.index.intersection(ref.columns).intersection(pd.Index(res.genes))
len(g), g

(1604,
 Index(['A4GNT', 'ABCA10', 'ABCA6', 'ABCA9', 'ABCB4', 'ABCC8', 'ABCC9', 'ABCD2', 'ABCG2', 'ABHD17C',
        ...
        'ZFHX4', 'ZG16', 'ZG16B', 'ZNF366', 'ZNF469', 'ZNF541', 'ZNF683', 'ZNF80', 'ZNF831',
        'ZNF98'],
       dtype='object', length=1604))

In [31]:
Phi = (ref[g].div(ref[g].sum(axis=1), axis=0)).T.to_numpy()
Phi

array([[2.87136504e-06, 4.49897198e-06, 4.12572685e-06, ...,
        1.98074557e-06, 1.16621427e-04, 3.20296396e-06],
       [1.31576080e-04, 3.69915474e-05, 6.87621142e-06, ...,
        5.54608759e-06, 2.22047197e-05, 2.52354737e-06],
       [3.44057094e-04, 1.02976470e-04, 2.09036827e-05, ...,
        5.34801303e-06, 1.79130512e-05, 2.23236882e-06],
       ...,
       [1.01342296e-06, 1.14973729e-05, 1.65029074e-06, ...,
        5.94223671e-07, 2.05253711e-06, 9.70595141e-08],
       [3.88478800e-06, 9.99771552e-06, 4.95087222e-06, ...,
        2.57496924e-06, 3.73188566e-06, 2.91178542e-07],
       [6.58724921e-06, 4.49897198e-06, 2.20038765e-06, ...,
        1.18844734e-06, 3.48931309e-05, 9.99712995e-06]])

In [33]:
print("cond(Phi):", np.linalg.cond(Phi))
pd.DataFrame(np.corrcoef(Phi.T), index=res.states, columns=res.states).round(2)

cond(Phi): 11.389287976168378


,Fibroblast cell,Stellate cell,Macrophage cell,Endothelial cell,T cell,B cell,Ductal cell type 2,Endocrine cell,Ductal cell type 1,Acinar cell
Fibroblast cell,1.00,0.43,0.28,0.17,0.05,0.16,0.20,0.10,0.10,0.02
Stellate cell,0.43,1.00,0.43,0.37,0.09,0.25,0.30,0.27,0.23,0.07
Macrophage cell,0.28,0.43,1.00,0.30,0.10,0.42,0.54,0.07,0.17,0.03
Endothelial cell,0.17,0.37,0.30,1.00,0.08,0.23,0.28,0.62,0.47,0.25
T cell,0.05,0.09,0.10,0.08,1.00,0.82,0.08,0.08,0.05,0.01
B cell,0.16,0.25,0.42,0.23,0.82,1.00,0.28,0.17,0.15,0.05
Ductal cell type 2,0.20,0.30,0.54,0.28,0.08,0.28,1.00,0.14,0.24,0.16
Endocrine cell,0.10,0.27,0.07,0.62,0.08,0.17,0.14,1.00,0.46,0.04
Ductal cell type 1,0.10,0.23,0.17,0.47,0.05,0.15,0.24,0.46,1.00,0.22
Acinar cell,0.02,0.07,0.03,0.25,0.01,0.05,0.16,0.04,0.22,1.00


Condition number above ~10³ means the states aren't separable from this reference and the NNLS disagreement is telling you about the reference, not the engine. 

Comparing at the coarse type level (res.theta_type) rather than the fine state level sidesteps this and is usually the fairer test.